In [11]:
# ── Imports ───────────────────────────────────────────────────────
from functools import partial
import functools
import datetime
import time
import os

import pandas as pd
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
import pyro.infer.autoguide as autoguide
from pyro.optim import Adam
from scipy.stats import spearmanr
from sklearn.preprocessing import RobustScaler
import polars as pl

from pybaseballstats.statcast import pitch_by_pitch_data
from pybaseballstats.utils.retrosheet_utils import _get_people_data

# ── Season Config ─────────────────────────────────────────────────
PRED_SEASON = 2026
TEST_SEASON = PRED_SEASON - 1

DECAY_FEATURES = ['xwOBA', 'BB%', 'K%', 'BsR']
LAMBDA         = 0.5 ** (1/3)

CONT_FEATURES = (
    [f'{f}_decay' for f in ['xwOBA', 'K%', 'BB%', 'BsR']] +
    ['age', 'age_sq', 'experience', 'delta_PA']
)
POSITIONS = ['C', '1B', '2B', '3B', 'SS', 'OF', 'DH', 'UNK']

SEASON_GAMES = {yr: (60 if yr == 2020 else 162) for yr in range(2019, TEST_SEASON + 1)}

_STATCAST_CACHE_DIR = os.path.join(
    os.path.dirname(os.path.abspath('.')), 'draft-tools', 'data', 'statcast_cache'
)

CURRENT_YEAR = 2026


# ── Statcast helpers ──────────────────────────────────────────────
def _fetch_hitter_season_raw(yr):
    # shared cache with pitcher notebook
    os.makedirs(_STATCAST_CACHE_DIR, exist_ok=True)
    cache_path = os.path.join(_STATCAST_CACHE_DIR, f'{yr}.parquet')

    is_current = (yr == CURRENT_YEAR)
    if os.path.exists(cache_path):
        if not is_current:
            return pd.read_parquet(cache_path)
        age = (datetime.datetime.now() -
               datetime.datetime.fromtimestamp(os.path.getmtime(cache_path))).total_seconds()
        if age < 86400:
            return pd.read_parquet(cache_path)

    month_starts = [datetime.date(yr, m, 1) for m in [3, 4, 5, 6, 7, 8, 9, 10]]
    chunks = []
    for ms in month_starts:
        me = (ms.replace(day=28) + datetime.timedelta(days=4))
        me = me - datetime.timedelta(days=me.day)
        if is_current and ms > datetime.date.today():
            break
        end = min(me, datetime.date.today()) if is_current else me
        max_retries, delay = 3, 10
        for attempt in range(1, max_retries + 1):
            try:
                result = pitch_by_pitch_data(
                    start_date=ms.strftime('%Y-%m-%d'),
                    end_date=end.strftime('%Y-%m-%d'),
                    force_collect=True
                )
                if result is not None:
                    chunks.append(result.to_pandas())
                break
            except RuntimeError as e:
                if attempt < max_retries:
                    print(f'  [{yr}-{ms.month:02d}] attempt {attempt} failed, retrying in {delay}s: {e}')
                    time.sleep(delay)
                    delay *= 2
                else:
                    print(f'  [{yr}-{ms.month:02d}] all retries failed, skipping month')

    if not chunks:
        return pd.DataFrame()

    df = pd.concat(chunks, ignore_index=True)

    # coerce mixed object columns
    for col in df.select_dtypes(include='object').columns:
        converted = pd.to_numeric(df[col], errors='coerce')
        non_null = df[col].dropna()
        if len(non_null) > 0:
            n_numeric = pd.to_numeric(non_null, errors='coerce').notna().sum()
            if n_numeric / len(non_null) >= 0.5:
                df[col] = converted
            else:
                df[col] = df[col].where(df[col].notna(), other=None).astype(str).where(df[col].notna(), other=None)

    df.to_parquet(cache_path, index=False)
    return df


@functools.lru_cache(maxsize=1)
def _build_id_maps():
    # Returns mlbam_to_name dict. We use MLBAM as the player key throughout
    # to avoid gaps in the Chadwick key_fangraphs mapping.
    people = _get_people_data()
    df = (people
          .filter(pl.col('key_mlbam').is_not_null())
          .select(['key_mlbam', 'name_first', 'name_last'])
          .to_pandas())
    df['key_mlbam'] = df['key_mlbam'].astype(int)
    df['Name'] = (df['name_first'].fillna('') + ' ' + df['name_last'].fillna('')).str.strip().str.title()
    return dict(zip(df['key_mlbam'], df['Name']))


def _compute_batter_season(raw, yr):
    raw = raw.sort_values(['game_pk', 'at_bat_number', 'pitch_number'])

    PA_EVENTS = frozenset([
        'single', 'double', 'triple', 'home_run',
        'walk', 'hit_by_pitch', 'intent_walk',
        'strikeout', 'strikeout_double_play',
        'field_out', 'grounded_into_double_play', 'double_play',
        'force_out', 'fielders_choice', 'fielders_choice_out',
        'field_error', 'sac_fly', 'sac_bunt',
        'sac_fly_double_play', 'triple_play', 'catcher_interf',
    ])

    # SB/CS: detect from base-state changes between consecutive pitches.
    # Statcast pitch data often lacks explicit stolen_base event rows;
    # tracking which runner moved to the next base is more reliable.
    _g   = raw.groupby(['game_pk', 'at_bat_number'])
    _n1b = _g['on_1b'].shift(-1)
    _n2b = _g['on_2b'].shift(-1)
    _n3b = _g['on_3b'].shift(-1)
    _nbs = _g['bat_score'].shift(-1)

    # SB: runner appears on next base in following pitch of the same at-bat
    _sb = pd.concat([
        raw.loc[raw['on_1b'].notna() & (raw['on_1b'] == _n2b), 'on_1b'],
        raw.loc[raw['on_2b'].notna() & (raw['on_2b'] == _n3b), 'on_2b'],
    ]).astype(float).dropna().astype(int)

    # CS: runner gone from base and not on next base, no score change
    _cs = pd.concat([
        raw.loc[
            raw['on_1b'].notna() &
            (raw['on_1b'] != _n1b.fillna(-1)) &
            (raw['on_1b'] != _n2b.fillna(-1)) &
            (_nbs.fillna(-1) == raw['bat_score'].fillna(-2)), 'on_1b'],
        raw.loc[
            raw['on_2b'].notna() &
            (raw['on_2b'] != _n2b.fillna(-1)) &
            (raw['on_2b'] != _n3b.fillna(-1)) &
            (_nbs.fillna(-1) == raw['bat_score'].fillna(-2)), 'on_2b'],
    ]).astype(float).dropna().astype(int)

    sb_s = _sb.value_counts().rename('SB') if len(_sb) > 0 else pd.Series(dtype=int, name='SB')
    cs_s = _cs.value_counts().rename('CS') if len(_cs) > 0 else pd.Series(dtype=int, name='CS')

    pa_df = raw[raw['events'].isin(PA_EVENTS)].copy()
    pa_df['_rbi'] = (pa_df['post_bat_score'] - pa_df['bat_score']).clip(lower=0)

    # R: credit the RUNNER when they score, not the batter who drove them in.
    # For HR: batter scores + runners on base (in order from 3B).
    # For non-HR scoring plays: credit runners from 3B inward up to score delta.
    runs_dict: dict = {}
    _nan = float('nan')
    for row in pa_df.itertuples(index=False):
        ps = getattr(row, 'post_bat_score', _nan)
        bs = getattr(row, 'bat_score',      _nan)
        if ps != ps or bs != bs:
            continue
        delta = max(0, int(ps) - int(bs))
        if delta == 0:
            continue
        ev     = row.events
        batter = int(row.batter)
        if ev == 'home_run':
            runs_dict[batter] = runs_dict.get(batter, 0) + 1
            delta -= 1
        for attr in ('on_3b', 'on_2b', 'on_1b'):
            if delta <= 0:
                break
            runner = getattr(row, attr, None)
            if runner is None or runner != runner:
                continue
            r = int(runner)
            runs_dict[r] = runs_dict.get(r, 0) + 1
            delta -= 1

    r_s = pd.Series(runs_dict, name='R')

    # xwOBA per PA: include BB and HBP contributions (matches FanGraphs definition).
    # estimated_woba_using_speedangle is only populated for batted balls;
    # walks and HBP use approximate annual wOBA scale weights.
    _BB_W, _HBP_W = 0.69, 0.72
    pa_df['_xw_num'] = pa_df['estimated_woba_using_speedangle'].fillna(0.0)
    pa_df.loc[pa_df['events'].isin(['walk', 'intent_walk']), '_xw_num'] = _BB_W
    pa_df.loc[pa_df['events'] == 'hit_by_pitch',            '_xw_num'] = _HBP_W
    xwoba_s = (pa_df.groupby('batter')['_xw_num'].sum()
               / pa_df.groupby('batter')['events'].count()).rename('xwOBA')

    # Vectorized PA aggregation
    ev = pa_df['events']
    bool_cols = {
        'H':   ev.isin(['single','double','triple','home_run']),
        '_2B': ev == 'double',
        '_3B': ev == 'triple',
        'HR':  ev == 'home_run',
        'BB':  ev.isin(['walk','intent_walk']),
        'SO':  ev.isin(['strikeout','strikeout_double_play']),
        'HBP': ev == 'hit_by_pitch',
        'GDP': ev.isin(['grounded_into_double_play','double_play']),
    }
    for col, mask in bool_cols.items():
        pa_df[col] = mask.astype(int)

    season = pa_df.groupby('batter').agg(
        PA  = ('events', 'count'),
        H   = ('H',   'sum'),
        _2B = ('_2B', 'sum'),
        _3B = ('_3B', 'sum'),
        HR  = ('HR',  'sum'),
        BB  = ('BB',  'sum'),
        SO  = ('SO',  'sum'),
        HBP = ('HBP', 'sum'),
        GDP = ('GDP', 'sum'),
        RBI = ('_rbi', 'sum'),
        Age = ('age_bat', 'first'),
    ).reset_index().rename(columns={'_2B': '2B', '_3B': '3B'})
    season = season.join(xwoba_s, on='batter')

    season = (season
              .join(sb_s, on='batter')
              .join(cs_s, on='batter')
              .join(r_s,  on='batter'))
    season['SB'] = season['SB'].fillna(0).astype(int)
    season['CS'] = season['CS'].fillna(0).astype(int)
    season['R']  = season['R'].fillna(0).astype(int)

    season = season[season['PA'] >= 250].copy()
    season['Season'] = yr
    season['BsR'] = 0.2 * season['SB'] - 0.44 * season['CS'] - 0.3 * season['GDP']
    season['K%']  = season['SO'] / season['PA'].clip(lower=1)
    season['BB%'] = season['BB'] / season['PA'].clip(lower=1)

    return season.reset_index(drop=True)


# ── Data Pipeline ─────────────────────────────────────────────────
def pull_seasons(seasons):
    dfs = []
    for yr in seasons:
        print(f'Fetching {yr}...')
        raw = _fetch_hitter_season_raw(yr)
        if raw.empty:
            print(f'  No data for {yr}')
            continue
        # Regular season only — exclude spring training ('S'), exhibitions ('E'),
        # All-Star ('A'), and postseason ('D','L','F','W').
        if 'game_type' in raw.columns:
            raw = raw[raw['game_type'] == 'R']
        raw = raw[raw['events'].notna()]
        stats = _compute_batter_season(raw, yr)
        dfs.append(stats)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


def _derive_positions_from_statcast(raw):
    # Infer position from which fielder slot each player appeared in.
    # fielder_2=C, 3=1B, 4=2B, 5=3B, 6=SS, 7/8/9=OF
    fielder_map = {
        'fielder_2': 'C',  'fielder_3': '1B', 'fielder_4': '2B',
        'fielder_5': '3B', 'fielder_6': 'SS',
        'fielder_7': 'OF', 'fielder_8': 'OF', 'fielder_9': 'OF',
    }
    rows = []
    for col, pos in fielder_map.items():
        if col not in raw.columns:
            continue
        sub = (raw[['game_pk', col]].dropna(subset=[col])
               .drop_duplicates()
               .copy())
        sub['mlbam'] = sub[col].astype(int)
        sub['Pos'] = pos
        rows.append(sub[['mlbam', 'Pos']])

    if not rows:
        return pd.DataFrame(columns=['batter', 'Inn', 'Pos'])

    all_pos = pd.concat(rows, ignore_index=True)
    counts = all_pos.groupby(['mlbam', 'Pos']).size().reset_index(name='Inn')

    counts['batter'] = counts['mlbam'].astype(int)
    return counts[['batter', 'Inn', 'Pos']]


def pull_recent_fielding(season):
    raw = _fetch_hitter_season_raw(season)
    if raw.empty:
        return pd.DataFrame(columns=['batter', 'Inn', 'Pos'])
    return _derive_positions_from_statcast(raw)


def add_fantasy_points(df):
    df = df.copy()
    df['1B'] = df['H'] - df['2B'] - df['3B'] - df['HR']
    df['fantasy_pts'] = (
        df['R']    * 0.75 +
        df['1B']   * 1.0  +
        df['2B']   * 1.5  +
        df['3B']   * 2.0  +
        df['HR']   * 3.0  +
        df['RBI']  * 0.75 +
        df['BB']   * 1.0  +
        df['SO']   * -0.5 +
        df['HBP']  * 1.0  +
        df['SB']   * 1.0  +
        df['CS']   * -2.0 +
        df['GDP']  * -2.0
    )
    df['fantasy_pts_per_PA'] = df['fantasy_pts'] / df['PA'].clip(lower=1)
    return df


def add_primary_pos(df, fld_df):
    if fld_df.empty:
        df = df.copy()
        df['Primary_Pos'] = 'UNK'
        return df
    primary_pos = (
        fld_df.groupby('batter')
        .apply(lambda x: x.loc[x['Inn'].idxmax(), 'Pos'])
        .reset_index(name='Primary_Pos')
    )
    return df.merge(primary_pos, on='batter', how='left').fillna({'Primary_Pos': 'UNK'})


def _build_rows(df, include_target=True):
    rows = []
    for pid in df['batter'].unique():
        pdf = df[df['batter'] == pid].sort_values('Season')
        target_seasons = pdf['Season'].values if include_target else [pdf['Season'].max() + 1]

        for target_season in target_seasons:
            past = pdf[pdf['Season'] < target_season].tail(3)
            if len(past) == 0:
                continue

            w = np.array([LAMBDA ** (len(past) - 1 - j) for j in range(len(past))])
            w /= w.sum()

            # normalize counting stats to per-162 game rate
            past = past.copy()
            season_scale = np.array([162 / SEASON_GAMES.get(s, 162) for s in past['Season'].values])
            past['BsR'] = past['BsR'] * season_scale

            row = {}
            for feat in DECAY_FEATURES:
                vals = past[feat].values.astype(float)
                ww = w.copy()
                ww[np.isnan(vals)] = 0
                row[f'{feat}_decay'] = np.nan if ww.sum() == 0 else (vals * (ww / ww.sum())).sum()

            row['Primary_Pos_recent'] = past['Primary_Pos'].iloc[-1]
            row['age']        = past['Age'].iloc[-1] + 1
            row['age_sq']     = row['age'] ** 2
            row['experience'] = len(past)
            pa_per_162 = past['PA'].values / np.array([SEASON_GAMES.get(s, 162) for s in past['Season'].values]) * 162
            row['delta_PA'] = (pa_per_162[-1] - pa_per_162[-2]) if len(past) >= 2 else 0.0

            pos = row['Primary_Pos_recent']
            for p in POSITIONS:
                row[p] = 1.0 if pos == p else 0.0

            row['player'] = pid
            row['season'] = target_season

            if include_target:
                tgt = pdf[pdf['Season'] == target_season].iloc[0]
                sf  = 162 / SEASON_GAMES.get(target_season, 162)
                row['target_fantasy_pts_per_PA'] = tgt['fantasy_pts_per_PA']
                row['target_PA']                 = tgt['PA'] * sf

            rows.append(row)
    return pd.DataFrame(rows)


def build_dataset(start_year, end_year):
    df     = pull_seasons(list(range(start_year, end_year + 1)))
    fld_df = pull_recent_fielding(end_year)
    df     = add_fantasy_points(df)
    df     = add_primary_pos(df, fld_df)
    return _build_rows(df, include_target=True)


def build_prediction_dataset(end_year):
    df     = pull_seasons(list(range(end_year - 2, end_year + 1)))
    fld_df = pull_recent_fielding(end_year)
    df     = add_fantasy_points(df)
    df     = add_primary_pos(df, fld_df)
    return _build_rows(df, include_target=False)


def build_dataloaders(df, player_to_idx=None, scaler=None):
    if player_to_idx is None:
        player_to_idx = {pid: idx for idx, pid in enumerate(df['player'].unique())}
    else:
        max_idx = max(player_to_idx.values())
        for pid in df['player'].unique():
            if pid not in player_to_idx:
                max_idx += 1
                player_to_idx[pid] = max_idx

    player_idx = torch.tensor(
        [player_to_idx[pid] for pid in df['player'].values], dtype=torch.long
    )

    # scale continuous features
    X = df[CONT_FEATURES].values
    if scaler is None:
        scaler = RobustScaler()
        X = scaler.fit_transform(X)
    else:
        X = scaler.transform(X)

    data = {
        'player_idx': player_idx,
        'X_cont':     torch.tensor(X, dtype=torch.float),
        'pos_onehot': torch.tensor(df[POSITIONS].values, dtype=torch.float),
        'n_players':  len(player_to_idx),
    }
    if 'target_fantasy_pts_per_PA' in df.columns:
        data['y_pts_per_pa'] = torch.tensor(
            df['target_fantasy_pts_per_PA'].values, dtype=torch.float)
        data['y_pa'] = torch.tensor(
            df['target_PA'].values, dtype=torch.float)

    return data, player_to_idx, scaler


def add_player_names(df):
    additional_lookup = {
        25477: 'Ben Rice',
        29576: 'Isaac Collins',
        29518: 'James Wood',
        27464: 'Dillon Dingler',
        31781: 'Jackson Holliday',
        31595: 'Brooks Lee',
        24598: 'Addison Barger',
        29794: 'Kyle Manzardo',
        24816: 'Andy Pages',
        28253: 'Javier Sanoja',
        23395: 'Eric Wagaman',
        31431: 'Jordan Beck',
        22857: 'Wenceel Perez',
        19722: 'Carlos Narvaez',
        26540: 'Angel Martinez',
        28312: 'Coby Mayo',
        25180: 'Daniel Schneemann',
        33541: 'Dylan Crews',
        31394: 'Brooks Baldwin',
        26374: 'Kameron Misner',
        25782: 'Pedro Pages',
        29592: 'Connor Norby',
        29575: 'Trey Sweeney',
        27883: 'Thomas Saggese',
        24605: 'Michael Helman',
        29478: 'Carter Jensen',
        29995: 'Daylen Lile',
        25999: 'Victor Mesa Jr.',
        31812: 'Roman Anthony',
        31505: 'Sal Stewart',
        21496: 'Brandon Lockridge',
    }
    MANUAL = {
        665742: 'Juan Soto',
        608070: 'José Ramírez',
        700250: 'Ben Rice',
        624413: 'Pete Alonso',
        701762: 'Nick Kurtz',
        701350: 'Roman Anthony',
        695734: 'Daylen Lile',
    }
    mlbam_to_name = _build_id_maps()
    df = df.copy()
    df['name'] = df['player'].apply(
        lambda x: MANUAL.get(int(x)) or mlbam_to_name.get(int(x), str(int(x)))
    )
    return df


# ── Models ────────────────────────────────────────────────────────
def model_pts_per_pa(player_idx, X_cont, y=None, n_players=500):
    n_cont = X_cont.shape[1]

    alpha   = pyro.sample('alpha',   dist.Normal(0.5, 1.0))
    beta    = pyro.sample('beta',    dist.StudentT(3., torch.zeros(n_cont), torch.ones(n_cont)).to_event(1))
    sigma   = pyro.sample('sigma',   dist.HalfNormal(0.2))
    sigma_u = pyro.sample('sigma_u', dist.HalfNormal(0.2))

    with pyro.plate('players', n_players):
        u_players = pyro.sample('player_effect', dist.Normal(0., sigma_u))

    with pyro.plate('obs', X_cont.shape[0]):
        safe_idx = player_idx.clamp(0, n_players - 1)
        u = u_players[safe_idx] * (player_idx < n_players).float()
        mu = alpha + u + (X_cont * beta).sum(-1)
        pyro.sample('y', dist.Normal(mu, sigma), obs=y)


def model_pa(player_idx, X_cont, pos_onehot, y=None, n_players=500):
    n_cont = X_cont.shape[1]

    alpha    = pyro.sample('alpha',    dist.Normal(torch.log(torch.tensor(650.)), 0.5))
    beta     = pyro.sample('beta',     dist.StudentT(3., torch.zeros(n_cont),              torch.ones(n_cont)*0.3).to_event(1))
    beta_pos = pyro.sample('beta_pos', dist.StudentT(3., torch.zeros(pos_onehot.shape[1]), torch.ones(pos_onehot.shape[1])*0.3).to_event(1))
    sigma    = pyro.sample('sigma',    dist.HalfNormal(0.1))
    sigma_u  = pyro.sample('sigma_u',  dist.HalfNormal(0.2))

    with pyro.plate('players', n_players):
        u_players = pyro.sample('player_effect', dist.Normal(0., sigma_u))

    with pyro.plate('obs', X_cont.shape[0]):
        safe_idx = player_idx.clamp(0, n_players - 1)
        u = u_players[safe_idx] * (player_idx < n_players).float()
        mu = alpha + u + (pos_onehot * beta_pos).sum(-1) + (X_cont * beta).sum(-1)
        pyro.sample('y_log', dist.Normal(mu, sigma),
                    obs=y.log() if y is not None else None)


# ── Training ──────────────────────────────────────────────────────
def train_model(model_fixed, args_fn, data, n_steps=1000, lr=0.01, n_samples=500):
    pyro.clear_param_store()
    guide = autoguide.AutoDiagonalNormal(model_fixed)
    svi = SVI(model_fixed, guide, Adam({'lr': lr}), Trace_ELBO(num_particles=4))
    losses = []
    for step in range(n_steps):
        loss = svi.step(*args_fn(data))
        losses.append(loss)
        if step % 200 == 0:
            print(f'  step {step:4d}  loss={loss:.1f}')
    print(f'  final loss={losses[-1]:.1f}, initial loss={losses[0]:.1f}')
    posterior_samples = Predictive(guide, num_samples=n_samples)(*args_fn(data))

    medians = guide.median()
    for k, v in medians.items():
        print(f'  {k}: {v.detach().flatten()[:5].numpy()}')

    exclude = {'_AutoDiagonalNormal_latent'}
    posterior_samples = {
        k: v.squeeze(1)
        for k, v in posterior_samples.items()
        if k not in exclude
    }
    print("posterior keys:", list(posterior_samples.keys()))
    print("sample shapes:", {k: v.shape for k, v in posterior_samples.items()})
    return posterior_samples


def train(data):
    n_players = data['n_players']
    model_pts_fixed = partial(model_pts_per_pa, n_players=n_players)
    model_pa_fixed  = partial(model_pa,         n_players=n_players)

    print('Training pts_per_pa...')
    pts_samples = train_model(
        model_pts_fixed,
        lambda d: (d['player_idx'], d['X_cont'], d['y_pts_per_pa']),
        data, n_steps=3000, lr=0.01
    )

    print('Training pa...')
    pa_samples = train_model(
        model_pa_fixed,
        lambda d: (d['player_idx'], d['X_cont'], d['pos_onehot'], d['y_pa']),
        data, n_steps=2000, lr=0.01
    )

    return pts_samples, pa_samples, model_pts_fixed, model_pa_fixed


def predict(pts_samples, pa_samples, model_pts_fixed, model_pa_fixed, data, n_samples=500):
    pred_pts = Predictive(model_pts_fixed, posterior_samples=pts_samples, num_samples=n_samples)(
        data['player_idx'], data['X_cont']
    )['y']

    pred_pa = Predictive(model_pa_fixed, posterior_samples=pa_samples, num_samples=n_samples)(
        data['player_idx'], data['X_cont'], data['pos_onehot']
    )['y_log'].clamp(max=torch.log(torch.tensor(750.))).exp()

    tot = pred_pts * pred_pa

    return {
        'mean_pts':        tot.mean(0).detach().numpy(),
        'std_pts':         tot.std(0).detach().numpy(),
        'mean_pts_per_pa': pred_pts.mean(0).detach().numpy(),
        'mean_pa':         pred_pa.mean(0).detach().numpy(),
    }


def risk_adjusted_score(mean, std, alpha=0.5):
    return mean / (std ** alpha)


# ── Evaluation ────────────────────────────────────────────────────
def evaluate(data):
    train_raw = data[data['season'] < TEST_SEASON].copy()
    test_raw  = data[data['season'] == TEST_SEASON].copy().reset_index(drop=True)

    player_to_idx = {pid: idx for idx, pid in enumerate(train_raw['player'].unique())}
    train_data, player_to_idx, scaler = build_dataloaders(train_raw, player_to_idx)
    test_data,  _, _                  = build_dataloaders(test_raw, player_to_idx, scaler=scaler)

    pts_samples, pa_samples, model_pts_fixed, model_pa_fixed = train(train_data)

    preds = predict(pts_samples, pa_samples, model_pts_fixed, model_pa_fixed, test_data)
    pred_tot        = preds['mean_pts']
    pred_pts_per_pa = preds['mean_pts_per_pa']
    pred_pa         = preds['mean_pa']

    test_raw['pred_fantasy_pts'] = pred_tot
    test_raw['pred_pts_per_pa']  = pred_pts_per_pa
    test_raw['pred_pa']          = pred_pa
    test_raw['true_fantasy_pts'] = test_raw['target_fantasy_pts_per_PA'] * test_raw['target_PA']

    mlbam_to_name = _build_id_maps()
    test_raw['name'] = test_raw['player'].apply(lambda x: mlbam_to_name.get(int(x), str(x)))

    corr_tot,        _ = spearmanr(pred_tot, test_raw['true_fantasy_pts'])
    corr_pts_per_pa, _ = spearmanr(pred_pts_per_pa, test_raw['target_fantasy_pts_per_PA'])
    corr_pa,         _ = spearmanr(pred_pa, test_raw['target_PA'])

    print(f'Total pts  Spearman: {corr_tot:.4f}')
    print(f'pts_per_PA Spearman: {corr_pts_per_pa:.4f}')
    print(f'PA         Spearman: {corr_pa:.4f}')

    return {
        'corr_tot':        corr_tot,
        'corr_pts_per_pa': corr_pts_per_pa,
        'corr_pa':         corr_pa,
        'results':         test_raw,
        'pts_samples':     pts_samples,
        'pa_samples':      pa_samples,
        'model_pts_fixed': model_pts_fixed,
        'model_pa_fixed':  model_pa_fixed,
        'player_to_idx':   player_to_idx,
        'scaler':          scaler,
    }


# ── Draft Rankings ────────────────────────────────────────────────
def get_draft_rankings(end_year=TEST_SEASON, alpha=0.5):
    pred_df  = build_prediction_dataset(end_year).dropna(subset=CONT_FEATURES)
    full_raw = build_dataset(2019, end_year).dropna(subset=CONT_FEATURES)

    player_to_idx = {pid: idx for idx, pid in enumerate(full_raw['player'].unique())}
    full_data, player_to_idx, scaler = build_dataloaders(full_raw, player_to_idx)
    pred_data, _, _                  = build_dataloaders(pred_df, player_to_idx, scaler=scaler)

    pts_samples, pa_samples, model_pts_fixed, model_pa_fixed = train(full_data)
    preds = predict(pts_samples, pa_samples, model_pts_fixed, model_pa_fixed, pred_data)

    pred_df = add_player_names(pred_df)

    pred_df = pred_df.copy()
    pred_df['mean_pts']        = preds['mean_pts']
    pred_df['std_pts']         = preds['std_pts']
    pred_df['mean_pts_per_pa'] = preds['mean_pts_per_pa']
    pred_df['pred_pa']         = preds['mean_pa']
    pred_df['risk_score']      = risk_adjusted_score(preds['mean_pts'], preds['std_pts'], alpha=alpha)

    RANKING_COLS = [
        'name', 'player', 'Primary_Pos_recent',
        'mean_pts', 'std_pts', 'risk_score',
        'mean_pts_per_pa', 'pred_pa',
        'age', 'experience',
        'xwOBA_decay', 'BB%_decay', 'K%_decay', 'BsR_decay',
    ]

    return (pred_df
            .sort_values('risk_score', ascending=False)
            .reset_index(drop=True)
            [RANKING_COLS]
    )


# ── Entry Point ───────────────────────────────────────────────────
# if __name__ == '__main__':
#     print('Building dataset...')
#     data = build_dataset(2019, TEST_SEASON)
#     data = data.dropna(subset=CONT_FEATURES)

    # print('Evaluating...')
    # results = evaluate(data)


In [12]:
# # ── Debug: check specific players in the pipeline ────────────────
# WATCH = {
#     592450: 'Aaron Judge',
#     660271: 'Shohei Ohtani',
#     608070: 'Jose Ramirez',
#     665742: 'Juan Soto',
# }

# mlbam_to_name = _build_id_maps()
# print("=== ID mapping (MLBAM -> name) ===")
# for mlbam, name in WATCH.items():
#     print(f"  {name}: mlbam={mlbam} -> '{mlbam_to_name.get(mlbam, 'NOT FOUND')}'")

# print()
# print("=== Computed season stats (2025) ===")
# raw = _fetch_hitter_season_raw(2025)
# stats_2025 = _compute_batter_season(raw, 2025)
# for mlbam, name in WATCH.items():
#     row = stats_2025[stats_2025['IDfg'] == mlbam]
#     if row.empty:
#         print(f"  {name}: NOT FOUND in computed stats")
#     else:
#         r = row.iloc[0]
#         print(f"  {name}: PA={r.PA}, HR={r.HR}, BB={r.BB}, SO={r.SO}, "
#               f"R={r.R}, RBI={r.RBI}, SB={r.SB}, xwOBA={r.xwOBA:.3f}")

# print()
# print("=== In prediction dataset (features) ===")
# pred_df_raw = build_prediction_dataset(TEST_SEASON)
# for mlbam, name in WATCH.items():
#     row = pred_df_raw[pred_df_raw['player'] == mlbam]
#     if row.empty:
#         print(f"  {name}: NOT IN pred_df")
#     else:
#         r = row.iloc[0]
#         print(f"  {name}: xwOBA_decay={r.xwOBA_decay:.3f}, BB%_decay={r['BB%_decay']:.3f}, "
#               f"K%_decay={r['K%_decay']:.3f}, age={r.age}, pos={r.Primary_Pos_recent}")


In [13]:
rankings = get_draft_rankings(alpha = 0.05)
rankings.to_csv('./outputs/hitter_rankings.csv', index=False)

Fetching 2023...
Fetching 2024...
Fetching 2025...


/var/folders/_y/9fww9bxx4qlc9wkrxyt0_6h40000gn/T/ipykernel_83488/3033179830.py:341: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['Inn'].idxmax(), 'Pos'])


Fetching 2019...
Fetching 2020...
Fetching 2021...
Fetching 2022...
Fetching 2023...
Fetching 2024...
Fetching 2025...


/var/folders/_y/9fww9bxx4qlc9wkrxyt0_6h40000gn/T/ipykernel_83488/3033179830.py:341: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['Inn'].idxmax(), 'Pos'])


Training pts_per_pa...
  step    0  loss=9079.3
  step  200  loss=-249.1
  step  400  loss=-490.1
  step  600  loss=-718.5
  step  800  loss=-804.2
  step 1000  loss=-848.2
  step 1200  loss=-844.0
  step 1400  loss=-867.9
  step 1600  loss=-876.2
  step 1800  loss=-881.0
  step 2000  loss=-878.0
  step 2200  loss=-878.2
  step 2400  loss=-881.6
  step 2600  loss=-879.2
  step 2800  loss=-889.9
  final loss=-882.5, initial loss=9079.3
  alpha: [0.4379925]
  beta: [ 0.03310435 -0.01594393  0.01972244 -0.00468009  0.0062867 ]
  sigma: [0.06772807]
  sigma_u: [0.04121546]
  player_effect: [ 0.0717505  -0.03023171  0.01273583 -0.00702544  0.02117004]
posterior keys: ['alpha', 'beta', 'sigma', 'sigma_u', 'player_effect']
sample shapes: {'alpha': torch.Size([500]), 'beta': torch.Size([500, 8]), 'sigma': torch.Size([500]), 'sigma_u': torch.Size([500]), 'player_effect': torch.Size([500, 373])}
Training pa...
  step    0  loss=93610.3
  step  200  loss=2301.3
  step  400  loss=661.2
  step  600

In [14]:
# ── Find players missing names in top 300 ────────────────────────
_pred = build_prediction_dataset(TEST_SEASON).dropna(subset=CONT_FEATURES)
_full = build_dataset(2019, TEST_SEASON).dropna(subset=CONT_FEATURES)
_p2i  = {pid: idx for idx, pid in enumerate(_full['player'].unique())}
_full_data, _p2i, _scaler = build_dataloaders(_full, _p2i)
_pred_data, _, _          = build_dataloaders(_pred, _p2i, scaler=_scaler)

_pts_s, _pa_s, _mpt, _mpa = train(_full_data)
_preds = predict(_pts_s, _pa_s, _mpt, _mpa, _pred_data)

_pred = add_player_names(_pred.copy())
_pred['mean_pts'] = _preds['mean_pts']
_pred = _pred.sort_values('mean_pts', ascending=False).head(300)

_mlbam_to_name = _build_id_maps()
missing = _pred[_pred['name'] == _pred['player'].astype(str)][['player', 'name', 'mean_pts']]
print(f"Players missing names in top 300: {len(missing)}")
for _, row in missing.iterrows():
    print(f"  mlbam={int(row.player)}  mean_pts={row.mean_pts:.1f}")


Fetching 2023...
Fetching 2024...
Fetching 2025...


/var/folders/_y/9fww9bxx4qlc9wkrxyt0_6h40000gn/T/ipykernel_83488/3033179830.py:341: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['Inn'].idxmax(), 'Pos'])


Fetching 2019...
Fetching 2020...
Fetching 2021...
Fetching 2022...
Fetching 2023...
Fetching 2024...
Fetching 2025...


/var/folders/_y/9fww9bxx4qlc9wkrxyt0_6h40000gn/T/ipykernel_83488/3033179830.py:341: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.loc[x['Inn'].idxmax(), 'Pos'])


Training pts_per_pa...
  step    0  loss=64378.1
  step  200  loss=-84.1
  step  400  loss=-330.1
  step  600  loss=-466.8
  step  800  loss=-592.9
  step 1000  loss=-617.2
  step 1200  loss=-712.5
  step 1400  loss=-719.2
  step 1600  loss=-781.9
  step 1800  loss=-773.3
  step 2000  loss=-857.3
  step 2200  loss=-854.1
  step 2400  loss=-861.7
  step 2600  loss=-880.7
  step 2800  loss=-869.0
  final loss=-870.9, initial loss=64378.1
  alpha: [0.44093257]
  beta: [ 2.7458370e-02 -1.7319091e-02  1.5773315e-02  4.4223631e-04
  9.8579738e-05]
  sigma: [0.07062906]
  sigma_u: [0.0408857]
  player_effect: [ 0.04255273 -0.03672342  0.01512568  0.00820969  0.01932634]
posterior keys: ['alpha', 'beta', 'sigma', 'sigma_u', 'player_effect']
sample shapes: {'alpha': torch.Size([500]), 'beta': torch.Size([500, 8]), 'sigma': torch.Size([500]), 'sigma_u': torch.Size([500]), 'player_effect': torch.Size([500, 373])}
Training pa...
  step    0  loss=41598.6
  step  200  loss=695.3
  step  400  loss=2

In [15]:
rankings.head(20)

,name,player,Primary_Pos_recent,mean_pts,std_pts,risk_score,mean_pts_per_pa,pred_pa,age,experience,xwOBA_decay,BB%_decay,K%_decay,BsR_decay
0,Aaron Judge,592450.0,OF,417.804688,86.122826,334.362366,0.673131,621.009644,34.0,3,0.474609,0.187158,0.250546,-4.629969
1,Juan Soto,665742.0,OF,402.832062,79.684906,323.634827,0.611078,659.352905,28.0,3,0.435879,0.180996,0.181072,-4.966489
2,Shohei Ohtani,660271.0,UNK,382.293762,85.488800,306.056610,0.636085,600.754944,32.0,3,0.435902,0.137636,0.240755,-2.627292
3,Vladimir Guerrero,665489.0,1B,368.306244,74.104736,296.972870,0.558601,659.297852,27.0,3,0.394674,0.108511,0.140252,-5.547590
4,Bobby Witt,677951.0,SS,368.314575,75.592369,296.684601,0.552309,666.387390,26.0,3,0.381275,0.070737,0.169351,-2.495150
5,José Ramírez,608070.0,3B,341.908569,78.844727,274.834564,0.564279,605.495667,34.0,3,0.352806,0.093851,0.112201,-3.233929
6,Ronald Acuña,660670.0,OF,342.484741,82.340683,274.701172,0.592889,578.026123,29.0,2,0.426340,0.144238,0.188594,-3.225724
7,Kyle Schwarber,656941.0,OF,339.320435,75.722870,273.305664,0.548001,619.345520,33.0,3,0.387397,0.157197,0.283111,-1.989851
8,Yordan Álvarez,670541.0,OF,339.206848,85.403069,271.575745,0.578456,584.798889,28.0,2,0.428366,0.122136,0.165482,-3.403512
9,Kyle Tucker,663656.0,OF,333.045380,85.298187,266.659088,0.591392,561.622864,29.0,3,0.385539,0.145076,0.148462,-2.696906
